# Задание 1

Датасеты:

1) **Emotion (HuggingFace)** эмоции: joy, sadness, anger, fear, love, surprise

2) **20 Newsgroups**, работаем только с 4мя классами: comp.sys.ibm.pc.hardware
comp.sys.mac.hardware
comp.graphics_
comp.windows.x

## Задание 1.1

Написать самим или разобрать код, запустить, посмотреть на 3х-5ти текстах как работает, найти
различия, которые возникают в результате различных подходов к предобработке текста. 

In [2]:
import nltk
import string
from datasets import load_dataset
import os


c:\Users\ysnxlmted\Projects\documents_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Создание папки для nltk данных, если её нет
nltk_data_dir = os.path.expanduser('nltk_data')
if not os.path.exists(nltk_data_dir):
    os.makedirs(nltk_data_dir)

# Добавление пути в nltk
nltk.data.path.append(nltk_data_dir)

# Загрузка всех необходимых ресурсов
resources = ['punkt', 'wordnet', 'omw-1.4', 'punkt_tab', 
             'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'stopwords']

for resource in resources:
    try:
        nltk.download(resource, download_dir=nltk_data_dir, quiet=False)
    except:
        print(f"Ресурс {resource} уже загружен или произошла ошибка")


[nltk_data] Downloading package punkt to nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [32]:
# Загрузка датасета Emotion (HuggingFace)
dataset = load_dataset("emotion")

In [34]:
# Возьмем первый текст
first_example = dataset["train"][0]
first_text = first_example["text"]

print("Исходный текст:")
print(first_text)

Исходный текст:
i didnt feel humiliated


In [35]:
# Шаг 1: Токенезация без доп. очистки
tokens = nltk.word_tokenize(first_text)

print("\nТокены (без дополнительной очистки):")
print(tokens)


Токены (без дополнительной очистки):
['i', 'didnt', 'feel', 'humiliated']


In [36]:
from nltk.corpus import wordnet
def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

In [37]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

In [38]:
# Функция предобработки со стеммингом
def preprocess_with_stemming(text):
    # Приведение к нижнему регистру
    text = text.lower()
    
    # Удаление знаков препинания
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Токенизация
    tokens = nltk.word_tokenize(text)
    
    # Применение стемминга
    stemmer = nltk.PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in tokens]

    filtered_tokens = []
    for i, token in enumerate(tokens):
        if token not in stop_words:
            filtered_tokens.append(stemmed_tokens[i])

    return filtered_tokens

# Функция предобработки с лемматизацией
def preprocess_with_lemmatization(text):
    # Приведение к нижнему регистру
    text = text.lower()
    
    # Токенизация
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    
    # Применение лемматизации
    lemmatizer = nltk.WordNetLemmatizer()
    lemmatized_tokens = []
    lemmatized_tokens = []
    for token, tag in tagged:
        if token not in stop_words and token not in string.punctuation:
            lemmatized = lemmatizer.lemmatize(token, get_wordnet_pos(tag))
            lemmatized_tokens.append(lemmatized)
    
    return lemmatized_tokens

In [39]:
# Сравним результаты стемминга и лемматизации
stemmed_result = preprocess_with_stemming(first_text)
lemmatized_result = preprocess_with_lemmatization(first_text)

print("\nТокены после стемминга:")
print(stemmed_result)
print("\nТокены после лемматизации:")
print(lemmatized_result)



Токены после стемминга:
['didnt', 'feel', 'humili']

Токены после лемматизации:
['didnt', 'feel', 'humiliate']


In [40]:
# Посмотрим другие примеры
other_examples = dataset["train"][1:5]
other_texts = other_examples["text"]

for i, text in enumerate(other_texts):
    print(f"Пример {i+1}:")
    print(text)
    print()

Пример 1:
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake

Пример 2:
im grabbing a minute to post i feel greedy wrong

Пример 3:
i am ever feeling nostalgic about the fireplace i will know that it is still on the property

Пример 4:
i am feeling grouchy



In [41]:
stemmed_result = [preprocess_with_stemming(text) for text in other_texts]
lemmatized_result = [preprocess_with_lemmatization(text) for text in other_texts]

for i in range(len(other_texts)):
    print(f"Пример {i+1}:")
    print(f"Stemming: {stemmed_result[i]}, length: {len(stemmed_result[i])}")
    print(f"Lemmatizing: {lemmatized_result[i]}, length: {len(lemmatized_result[i])}")
    print()

Пример 1:
Stemming: ['go', 'feel', 'hopeless', 'damn', 'hope', 'around', 'someon', 'care', 'awak'], length: 9
Lemmatizing: ['go', 'feel', 'hopeless', 'damned', 'hopeful', 'around', 'someone', 'care', 'awake'], length: 9

Пример 2:
Stemming: ['im', 'grab', 'minut', 'post', 'feel', 'greedi', 'wrong'], length: 7
Lemmatizing: ['im', 'grab', 'minute', 'post', 'feel', 'greedy', 'wrong'], length: 7

Пример 3:
Stemming: ['ever', 'feel', 'nostalg', 'fireplac', 'know', 'still', 'properti'], length: 7
Lemmatizing: ['ever', 'feel', 'nostalgic', 'fireplace', 'know', 'still', 'property'], length: 7

Пример 4:
Stemming: ['feel', 'grouchi'], length: 2
Lemmatizing: ['feel', 'grouchy'], length: 2



## Задание 2.3

Разные методы векторизации

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def vectorizer1(lemmatized_result):
    '''
    сразу засовываем лемматизированные тексты
    '''

    docs_as_strings = [''.join(tokens) for tokens in lemmatized_result]
    vectorizer = CountVectorizer(binary=True)
    
    X = vectorizer.fit_transform(docs_as_strings)
    vocab = vectorizer.get_feature_names_out()
    
    return X, vocab

def vectorizer2(raw_docs):
    '''
    подаем просто тексты, 
    внутри векторизатора используем наш токенезетор
    '''

    vectorizer = CountVectorizer(
        binary=True,
        tokenizer=preprocess_with_lemmatization,
        lowercase=False,
        token_pattern=None
    )
    
    X = vectorizer.fit_transform(raw_docs)
    vocab = vectorizer.get_feature_names_out()

    return X, vocab

def vectorizer3(raw_docs):
    '''
    полагаемся на исходный векторизатор
    '''

    vectorizer = CountVectorizer(binary=True)
    
    X = vectorizer.fit_transform(raw_docs)
    vocab = vectorizer.get_feature_names_out()

    return X, vocab

array(['about', 'am', 'and', 'around', 'awake', 'being', 'can', 'cares',
       'damned', 'ever', 'feel', 'feeling', 'fireplace', 'from', 'go',
       'grabbing', 'greedy', 'grouchy', 'hopeful', 'hopeless', 'im', 'is',
       'it', 'just', 'know', 'minute', 'nostalgic', 'on', 'post',
       'property', 'so', 'someone', 'still', 'that', 'the', 'to', 'who',
       'will', 'wrong'], dtype=object)

In [59]:
def vectorizers_outp(lemmatized_result, other_texts):
    vec1 = vectorizer1(lemmatized_result)
    vec2 = vectorizer2(other_texts)
    vec3 = vectorizer3(other_texts)
    print("СРАВНЕНИЕ ВЕКТОРИЗАТОРОВ")

    print("1st")
    print(f"Словарь: {vec1[1]}, \n {vec1[0].toarray()}")

    print("2st")
    print(f"Словарь: {vec2[1]}, \n {vec2[0].toarray()}")

    print("3st")
    print(f"Словарь: {vec3[1]}, \n {vec3[0].toarray()}")

In [60]:
vectorizers_outp(lemmatized_result, other_texts)

СРАВНЕНИЕ ВЕКТОРИЗАТОРОВ
1st
Словарь: ['everfeelnostalgicfireplaceknowstillproperty' 'feelgrouchy'
 'gofeelhopelessdamnedhopefularoundsomeonecareawake'
 'imgrabminutepostfeelgreedywrong'], 
 [[0 0 1 0]
 [0 0 0 1]
 [1 0 0 0]
 [0 1 0 0]]
2st
Словарь: ['around' 'awake' 'care' 'damned' 'ever' 'feel' 'fireplace' 'go' 'grab'
 'greedy' 'grouchy' 'hopeful' 'hopeless' 'im' 'know' 'minute' 'nostalgic'
 'post' 'property' 'someone' 'still' 'wrong'], 
 [[1 1 1 1 0 1 0 1 0 0 0 1 1 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 1 0 0 1 1 0 0 0 1 0 1 0 1 0 0 0 1]
 [0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 0 1 0 1 0 1 0]
 [0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]]
3st
Словарь: ['about' 'am' 'and' 'around' 'awake' 'being' 'can' 'cares' 'damned' 'ever'
 'feel' 'feeling' 'fireplace' 'from' 'go' 'grabbing' 'greedy' 'grouchy'
 'hopeful' 'hopeless' 'im' 'is' 'it' 'just' 'know' 'minute' 'nostalgic'
 'on' 'post' 'property' 'so' 'someone' 'still' 'that' 'the' 'to' 'who'
 'will' 'wrong'], 
 [[0 0 1 1 1 1 1 1 1 0 0 1 0 1 1 0 0 0 1 1 0 

## Задание 1.2

Вывести на экран и посмотреть, какая разница между результатами стемминга и лемматизации,
показать конкретные случаи различия. Если требуется, можно вывести на экран больше текстов.

**Стемминг** (Stemming) — это грубый (правило-ориентированный) способ «обрезать» слово до
его «основы» (stem). Часто приводит к «искусственным» словам (например, running → run, но
caring → car).

**Лемматизация** (Lemmatization) — более «осмысленный» метод, использующий словари
(лексические базы) для преобразования слова в его «лемму» (форму, которая обычно приводится
как основная в словаре). Например, running → run, caring → care.

In [61]:
# Функция подсвечивает различия
from difflib import Differ

def highlight_differences(word1, word2):
    differ = Differ()
    result = list(differ.compare(word1, word2))
    return result

word1 = "menace"
word2 = "menac"

highlighted = highlight_differences(word1, word2)
print(f"Различия: {highlighted}")

Различия: ['  m', '  e', '  n', '  a', '  c', '- e']


In [62]:
# Функция выводит конкретные случаи различия в стемминге и лемматизации
def find_diffs(stemmed_result, lemmatized_result):
    for sr, lr in zip(stemmed_result, lemmatized_result):
        if sr == lr:
            pass
        else:
            print(f"Stemming: {sr}, Lemmatizing: {lr}")
            print(f"Difference: {highlight_differences(sr, lr)}")
            print()

In [63]:
for i in range(len(other_texts)):
    find_diffs(stemmed_result[i], lemmatized_result[i])

Stemming: damn, Lemmatizing: damned
Difference: ['  d', '  a', '  m', '  n', '+ e', '+ d']

Stemming: hope, Lemmatizing: hopeful
Difference: ['  h', '  o', '  p', '  e', '+ f', '+ u', '+ l']

Stemming: someon, Lemmatizing: someone
Difference: ['  s', '  o', '  m', '  e', '  o', '  n', '+ e']

Stemming: awak, Lemmatizing: awake
Difference: ['  a', '  w', '  a', '  k', '+ e']

Stemming: minut, Lemmatizing: minute
Difference: ['  m', '  i', '  n', '  u', '  t', '+ e']

Stemming: greedi, Lemmatizing: greedy
Difference: ['  g', '  r', '  e', '  e', '  d', '- i', '+ y']

Stemming: nostalg, Lemmatizing: nostalgic
Difference: ['  n', '  o', '  s', '  t', '  a', '  l', '  g', '+ i', '+ c']

Stemming: fireplac, Lemmatizing: fireplace
Difference: ['  f', '  i', '  r', '  e', '  p', '  l', '  a', '  c', '+ e']

Stemming: properti, Lemmatizing: property
Difference: ['  p', '  r', '  o', '  p', '  e', '  r', '  t', '- i', '+ y']

Stemming: grouchi, Lemmatizing: grouchy
Difference: ['  g', '  r', '  

## Задание 1.3

Есть высказывание, которое нужно проверить экспериментально, выводя результат на экран:

По умолчанию ```nltk.word_tokenize``` не удаляет знаки препинания, а просто выделяет их в
отдельные токены. Например, если у вас есть фраза "Hello, world!", то результатом будет
что-то вроде ["Hello", ",", "world", "!"].

Если вы хотите полностью исключить пунктуацию из своего набора токенов (часто это нужно
в задачах классификации, чтобы «чистый» текст без знаков препинания был входом для модели),
то: либо удаляете их до токенизации, либо удаляете их после токенизации, отфильтровывая
токены, которые состоят только из знаков препинания (или содержат их).


In [64]:
phrase = "Hello, World!"

tokens = nltk.word_tokenize(phrase)
print(tokens)

# 1ый вариант
clean_phrase = phrase.translate(str.maketrans('', '', string.punctuation))
clean_tokens_1 = nltk.word_tokenize(clean_phrase)
print(clean_tokens_1)

# 2ый вариант
clean_tokens_2 = [token.translate(str.maketrans('', '', string.punctuation)) for token in tokens]
clean_tokens_2 = [token for token in clean_tokens_2 if token]
print(clean_tokens_2)

['Hello', ',', 'World', '!']
['Hello', 'World']
['Hello', 'World']


## Задание 1.4

Повторить все тоже самое для датасета **IMDB Movie Reviews**

In [ ]:
# Загрузка 20 Newsgroups с 4 классами
dataset = load_dataset("SetFit/20_newsgroups", split="train")

categories = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.graphics",
    "comp.windows.x"
]

filtered_texts = [
    example["text"]
    for example in dataset
    if example["label_text"] in categories
][:5]

filtered_texts

Repo card metadata block was not found. Setting CardData to empty.


["A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with the procedure. Top speed attained, CPU rated speed,\nadd on cards and adapters, heat sinks, hour of usage per day, floppy disk\nfunctionality with 800 and 1.4 m floppies are especially requested.\n\nI will be summarizing in the next two days, so please add to the network\nknowledge base if you have done the clock upgrade and haven't answered this\npoll. Thanks.",
 'well folks, my mac plus finally gave up the ghost this weekend after\nstarting life as a 512k way back in 1985.  sooo, i\'m in the market for a\nnew machine a bit sooner than i intended to be...\n\ni\'m looking into picking up a powerbook 160 or maybe 180 and have a bunch\nof questions that (hopefully) somebody can answer:\n\n* does anybody know any dirt on when the next round of powerbook\nintroductions are expected?  i\'d heard the 185c was supposed

In [66]:
for i, text in enumerate(filtered_texts):
    print(f"Пример {i+1}:")
    print(text)
    print()

Пример 1:
A fair number of brave souls who upgraded their SI clock oscillator have
shared their experiences for this poll. Please send a brief message detailing
your experiences with the procedure. Top speed attained, CPU rated speed,
add on cards and adapters, heat sinks, hour of usage per day, floppy disk
functionality with 800 and 1.4 m floppies are especially requested.

I will be summarizing in the next two days, so please add to the network
knowledge base if you have done the clock upgrade and haven't answered this
poll. Thanks.

Пример 2:
well folks, my mac plus finally gave up the ghost this weekend after
starting life as a 512k way back in 1985.  sooo, i'm in the market for a
new machine a bit sooner than i intended to be...

i'm looking into picking up a powerbook 160 or maybe 180 and have a bunch
of questions that (hopefully) somebody can answer:

* does anybody know any dirt on when the next round of powerbook
introductions are expected?  i'd heard the 185c was supposed to 

In [67]:
for i, text in enumerate(filtered_texts):
    tokens = nltk.word_tokenize(text)
    print(f"Пример {i+1}")
    print(tokens)
    print()

Пример 1
['A', 'fair', 'number', 'of', 'brave', 'souls', 'who', 'upgraded', 'their', 'SI', 'clock', 'oscillator', 'have', 'shared', 'their', 'experiences', 'for', 'this', 'poll', '.', 'Please', 'send', 'a', 'brief', 'message', 'detailing', 'your', 'experiences', 'with', 'the', 'procedure', '.', 'Top', 'speed', 'attained', ',', 'CPU', 'rated', 'speed', ',', 'add', 'on', 'cards', 'and', 'adapters', ',', 'heat', 'sinks', ',', 'hour', 'of', 'usage', 'per', 'day', ',', 'floppy', 'disk', 'functionality', 'with', '800', 'and', '1.4', 'm', 'floppies', 'are', 'especially', 'requested', '.', 'I', 'will', 'be', 'summarizing', 'in', 'the', 'next', 'two', 'days', ',', 'so', 'please', 'add', 'to', 'the', 'network', 'knowledge', 'base', 'if', 'you', 'have', 'done', 'the', 'clock', 'upgrade', 'and', 'have', "n't", 'answered', 'this', 'poll', '.', 'Thanks', '.']

Пример 2
['well', 'folks', ',', 'my', 'mac', 'plus', 'finally', 'gave', 'up', 'the', 'ghost', 'this', 'weekend', 'after', 'starting', 'life',

In [69]:
stemmed_result = [preprocess_with_stemming(text) for text in filtered_texts]
lemmatized_result = [preprocess_with_lemmatization(text) for text in filtered_texts]

for i in range(len(filtered_texts)):
    print(f"Пример {i+1}:")
    print(f"Stemming: {stemmed_result[i]}, length: {len(stemmed_result[i])}")
    print(f"Lemmatizing: {lemmatized_result[i]}, length: {len(lemmatized_result[i])}")
    print()

Пример 1:
Stemming: ['fair', 'number', 'brave', 'soul', 'upgrad', 'si', 'clock', 'oscil', 'share', 'experi', 'poll', 'pleas', 'send', 'brief', 'messag', 'detail', 'experi', 'procedur', 'top', 'speed', 'attain', 'cpu', 'rate', 'speed', 'add', 'card', 'adapt', 'heat', 'sink', 'hour', 'usag', 'per', 'day', 'floppi', 'disk', 'function', '800', '14', 'floppi', 'especi', 'request', 'summar', 'next', 'two', 'day', 'pleas', 'add', 'network', 'knowledg', 'base', 'done', 'clock', 'upgrad', 'havent', 'answer', 'poll', 'thank'], length: 57
Lemmatizing: ['fair', 'number', 'brave', 'soul', 'upgrade', 'si', 'clock', 'oscillator', 'share', 'experience', 'poll', 'please', 'send', 'brief', 'message', 'detail', 'experience', 'procedure', 'top', 'speed', 'attain', 'cpu', 'rat', 'speed', 'add', 'card', 'adapter', 'heat', 'sink', 'hour', 'usage', 'per', 'day', 'floppy', 'disk', 'functionality', '800', '1.4', 'floppy', 'especially', 'request', 'summarize', 'next', 'two', 'day', 'please', 'add', 'network', 'k

In [71]:
for i in range(len(filtered_texts)):
    find_diffs(stemmed_result[i], lemmatized_result[i])

Stemming: upgrad, Lemmatizing: upgrade
Difference: ['  u', '  p', '  g', '  r', '  a', '  d', '+ e']

Stemming: oscil, Lemmatizing: oscillator
Difference: ['  o', '  s', '  c', '  i', '  l', '+ l', '+ a', '+ t', '+ o', '+ r']

Stemming: experi, Lemmatizing: experience
Difference: ['  e', '  x', '  p', '  e', '  r', '  i', '+ e', '+ n', '+ c', '+ e']

Stemming: pleas, Lemmatizing: please
Difference: ['  p', '  l', '  e', '  a', '  s', '+ e']

Stemming: messag, Lemmatizing: message
Difference: ['  m', '  e', '  s', '  s', '  a', '  g', '+ e']

Stemming: experi, Lemmatizing: experience
Difference: ['  e', '  x', '  p', '  e', '  r', '  i', '+ e', '+ n', '+ c', '+ e']

Stemming: procedur, Lemmatizing: procedure
Difference: ['  p', '  r', '  o', '  c', '  e', '  d', '  u', '  r', '+ e']

Stemming: rate, Lemmatizing: rat
Difference: ['  r', '  a', '  t', '- e']

Stemming: adapt, Lemmatizing: adapter
Difference: ['  a', '  d', '  a', '  p', '  t', '+ e', '+ r']

Stemming: usag, Lemmatizing: u

In [72]:
vectorizers_outp(lemmatized_result, filtered_texts)

СРАВНЕНИЕ ВЕКТОРИЗАТОРОВ
1st
Словарь: ['1208' '12mb' '15mb' '1chip' '1controlerchiprangeindeed0'
 '1existpcusesetscsi' '1go' '1interfacedrivemachinescsi' '1interfacethink'
 '1maybescsi' '1mode4' '1reach10mb' '1scsi' '20fastide' '20mb' '216' '232'
 '28' '2chipapplesalespersonsay' '2controllerchip4' '2controllerchip8'
 '2controllerchipscsi' '39' '44'
 '4floppyespeciallyrequestsummarizenexttwodaypleaseaddnetworkknowledgebasedoclockupgraden'
 '5mb' '6info' '6mb'
 '96scsifactpostnewsgroupmacibminfosheetavailableftpsumex' 'aim'
 'althoughscsitwicefastesdi'
 'anybodyheardrumorpricedroppowerbooklinelikeoneduo' 'bit'
 'bitmodemuchfastertruescsi' 'bitnoteincreasespeedmacquadrauseversionscsi'
 'compareversion'
 'compressedfileunlessboardinstalsincestacproductseemunlikelyholeautodoubler'
 'correctscsi'
 'dayworthtakedisksizemoneyhitgetactivedisplayrealizerealsubjectivequestion'
 'dhear185csupposemakeappearence'
 'diskdoublerrelateboardfixsadmakereluctantbuystac'
 'displayyealookgreatstore' 'dlikeg